# LimiX-2

A 400M-parameter transformer pretrained by Stable AI on synthetic tables (released 16 Sept 2026). Like TabPFN, it is not trained on our data: it reads the training rows and predicts the new rows in one forward pass. This is a first run with the default settings, no tuning.

**Setup, once, in the `AML` env:**

```
pip install "git+https://github.com/limix-ldm-ai/LimiX.git@b4d06eadca6f0cd58ae259034b02c86f4a905950"
```

- This is the official code (`LimiXPredictor`). The model can't be loaded with `transformers`.
- It adds `LimiX`, `kditransform`, `numba`, `llvmlite` and `nvtx`. `torch` and `numpy` stay as they are.
- The weights (`LimiX-2.ckpt`, 1.63 GB) are downloaded from Hugging Face on the first run and cached in `~/.cache/huggingface`, outside the repo.
- LimiX officially supports CUDA and CPU only. Here it runs on the M4 GPU through MPS. Its model code imports Triton, which has no macOS version; the model cell works around that.
- License: StableAI LimiX Non-Commercial License.

In [1]:
import importlib
import os
import sys
import time
from unittest.mock import MagicMock
import numpy as np
import pandas as pd
import torch
import config as limix_config          # LimiX's bundled inference configs (installed with the package)
from huggingface_hub import hf_hub_download
from inference.predictor import LimiXPredictor
from sklearn.metrics import r2_score

X = pd.read_csv("../X_train_log_knn.csv", index_col="id")
X_test = pd.read_csv("../X_test_log_knn.csv", index_col="id")
y = pd.read_csv("../y_train.csv", index_col="id").loc[X.index, "y"]
fold = pd.read_csv("../Stacking/folds.csv", index_col="id").loc[X.index, "fold"]
assert list(X_test.columns) == list(X.columns)   # LimiX takes numpy arrays, so the column order must match
print(f"train {X.shape} | test {X_test.shape} | y {y.shape}")

train (1152, 216) | test (776, 216) | y (1152,)


In [2]:
# Pretrained weights from Hugging Face (1.63 GB, downloaded once) and the default regression config

model_path = hf_hub_download(repo_id="stable-ai/LimiX-2", filename="LimiX-2.ckpt")
config_path = os.path.join(os.path.dirname(limix_config.__file__), "reg_default_noretrieval_v2.json")

# LimiX's model code imports Triton (NVIDIA-only GPU kernels, no macOS version) but only uses it on CUDA:
# on MPS its RMSNorm layers fall back to torch's F.rms_norm. So import that code once with a placeholder
# in place of Triton, then remove the placeholder.
sys.modules["triton"] = sys.modules["triton.language"] = MagicMock()
importlib.import_module("model.v2_0")
del sys.modules["triton"], sys.modules["triton.language"]

model = LimiXPredictor(device=torch.device("mps"), model_path=model_path, inference_config=config_path)

arch_version=2.0 -> inference.v2_0.predictor (inference.v2_0.predictor)
use limix v2.0


In [3]:
# Out-of-fold predictions on the shared folds (runs on the M4 GPU through MPS)

oof = pd.Series(np.nan, index=X.index, name="limix")
for f in range(5):
    train, val = fold != f, fold == f
    start = time.time()
    oof[val] = model.predict(X[train].to_numpy(), y[train].to_numpy(), X[val].to_numpy(), task_type="Regression")
    print(f"fold {f}: R2 {r2_score(y[val], oof[val]):.4f} ({time.time() - start:.0f} s)")

print(f"out-of-fold R2: {r2_score(y, oof):.4f}")
oof.to_csv("../Stacking/oof_limix.csv")

fold 0: R2 0.8353 (112 s)


fold 1: R2 0.8274 (108 s)


fold 2: R2 0.7669 (115 s)


fold 3: R2 0.8060 (108 s)


fold 4: R2 0.8069 (108 s)
out-of-fold R2: 0.8092


In [4]:
# Compare with TabPFN on the same folds: R2 per fold, and how similar the errors are
# (error correlation close to 1 = same errors, so LimiX would add little to a stack)

tabpfn = pd.concat({m: pd.read_csv(f"../Stacking/oof_{m}.csv", index_col="id").iloc[:, 0]
                    for m in ["tabpfn", "tabpfn_ft"]}, axis=1).loc[X.index]
preds = pd.concat([oof, tabpfn], axis=1)

per_fold = pd.DataFrame({m: [r2_score(y[fold == f], preds.loc[fold == f, m]) for f in range(5)] for m in preds})
per_fold.loc["out-of-fold"] = [r2_score(y, preds[m]) for m in preds]
display(per_fold.round(4))

for m in tabpfn:
    diff = per_fold.loc[range(5), "limix"] - per_fold.loc[range(5), m]
    print(f"limix - {m}: {diff.mean():+.4f} ± {diff.std() / np.sqrt(5):.4f} (SE), better in {(diff > 0).sum()}/5 folds | "
          f"error correlation {(oof - y).corr(tabpfn[m] - y):.3f}")

,limix,tabpfn,tabpfn_ft
0,0.8353,0.8074,0.8193
1,0.8274,0.7970,0.7912
2,0.7669,0.7267,0.7339
3,0.8060,0.7669,0.7690
4,0.8069,0.7828,0.7861
out-of-fold,0.8092,0.7772,0.7810


limix - tabpfn: +0.0323 ± 0.0031 (SE), better in 5/5 folds | error correlation 0.913
limix - tabpfn_ft: +0.0286 ± 0.0043 (SE), better in 5/5 folds | error correlation 0.922


In [5]:
# Full model: all 1152 training rows, predicting the 776 test rows (sample.csv format, for Stacking/stacking.ipynb)

pred_test = model.predict(X.to_numpy(), y.to_numpy(), X_test.to_numpy(), task_type="Regression")
pd.Series(pred_test, index=X_test.index, name="y").to_csv("submission_limix.csv")